### Strcutured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing . Langchain supports multiple schema types and methods for enforcing structred output

### Pydantic

Pydantic Models provide the richest feature set with field validations , descriptions and nested structures

In [ ]:
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model


In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel): # BaseModel lets you define a schema for your data.
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int =Field(description="The year movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The Rating of the movie")

In [ ]:
model_with_struct_output = model.with_structured_output(Movie , include_raw= True) # To see the original LLM unstructred response
model_with_struct_output

In [ ]:
response =model_with_struct_output.invoke("Give some info about the Movie Titanic")
response

### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel): # BaseModel lets you define a schema for your data.
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int 
    director:str
    rating:float| None=Field(description="The Rating of the movie")
    casr:list[Actor]
    genres: list[str]

In [ ]:
model_with_struct_nested_output = model.with_structured_output(MovieDetails , include_raw= True) # To see the original LLM unstructred response
model_with_struct_nested_output

In [ ]:
response =model_with_struct_nested_output.invoke("Give some info about the Movie Inception")
response

### TypedDict

TypedDict provides a simpler alertnative using Python's builtin typing .
Ideal when :
- No Runtime validation needed
- Faster Performance
- Passing onl internal data

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with the details"""
    title: Annotated[str, ..., " The title of the movie"]
    year:Annotated[int, ..., "The year movie was released"]
    director:Annotated[str, ..., "he director of the movie"]
    rating:Annotated[float, ..., "The Rating of the movie"]

In [ ]:
model_with_typedDcit= model.with_structured_output(MovieDict)
response =model_with_typedDcit.invoke("Give some info about the Movie Avengers")
response

In [ ]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict): # B
    title:str=Field(description="The tile of the movie") # Field() adds metadata, validation rules, and descriptions.
    year:int 
    director:str
    rating:float| None=Field(description="The Rating of the movie")
    casr:list[Actor]
    genres: list[str]

In [ ]:
model_with_struct_nested_output = model.with_structured_output(MovieDetails , include_raw= True) # To see the original LLM unstructred response
model_with_struct_nested_output

In [ ]:
response =model_with_struct_nested_output.invoke("Give some info about the Movie Inception")
response

In [ ]:
model.profile

### Dataclasses

A data class is a class typically containing mainly data, although there aren't really any restrictions . you can create it using @dataclass decorator

In [ ]:
from pydantic import BaseModel, Field

from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str=Field(description="Name of the Person")
    email: str=Field(description="Email of the Person")
    phone: str=Field(description="Phone number of the Person")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo

)

result= agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extarct the contact info from John Doe, john@example.com, 123-234-1212"
    }]
})

result["structured_response"]

In [ ]:
# using dataclass instead of Pydantic or TypedDict to get the structured output

In [ ]:
from dataclasses import dataclass
@dataclass
class ContactInfo(BaseModel):
    """Contact information of a person"""
    name: str=Field(description="Name of the Person")
    email: str=Field(description="Email of the Person")
    phone: str=Field(description="Phone number of the Person")

In [ ]:
agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo

)

result= agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extarct the contact info from John Doe, john@example.com, 123-234-1212"
    }]
})

result["structured_response"]